In [8]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import re
from bs4 import BeautifulSoup

import warnings
warnings.filterwarnings('ignore')

In [9]:
df = pd.read_csv('../train.csv')


In [10]:
new_df = df.sample(30000,random_state=2)

In [11]:
def preprocess(q):
    q = str(q).lower().strip()
    
    # Replace certain special characters with their string equivalents:
    q = q.replace('%', ' percent')
    q = q.replace('$', ' dollar ')
    q = q.replace('₹', ' rupee ')
    q = q.replace('€', ' euro ')
    q = q.replace('@', ' at ')


    # The pattern '[math]' appears around 900 times in the whole dataset.
    q = q.replace('[math]', '')

    # Replacing some numbers with string equivalents (not perfect, can be done better to account for more cases)
    q = q.replace(',000,000,000 ', 'b ')
    q = q.replace(',000,000 ', 'm ')
    q = q.replace(',000 ', 'k ')
    q = re.sub(r'([0-9]+)000000000', r'\1b', q)
    q = re.sub(r'([0-9]+)000000', r'\1m', q)
    q = re.sub(r'([0-9]+)000', r'\1k', q)

    # Decontracting words
    # https://en.wikipedia.org/wiki/Wikipedia%3aList_of_English_contractions
    # https://stackoverflow.com/a/19794953
    contractions = { 
    "ain't": "am not",
    "aren't": "are not",
    "can't": "can not",
    "can't've": "can not have",
    "'cause": "because",
    "could've": "could have",
    "couldn't": "could not",
    "couldn't've": "could not have",
    "didn't": "did not",
    "doesn't": "does not",
    "don't": "do not",
    "hadn't": "had not",
    "hadn't've": "had not have",
    "hasn't": "has not",
    "haven't": "have not",
    "he'd": "he would",
    "he'd've": "he would have",
    "he'll": "he will",
    "he'll've": "he will have",
    "he's": "he is",
    "how'd": "how did",
    "how'd'y": "how do you",
    "how'll": "how will",
    "how's": "how is",
    "i'd": "i would",
    "i'd've": "i would have",
    "i'll": "i will",
    "i'll've": "i will have",
    "i'm": "i am",
    "i've": "i have",
    "isn't": "is not",
    "it'd": "it would",
    "it'd've": "it would have",
    "it'll": "it will",
    "it'll've": "it will have",
    "it's": "it is",
    "let's": "let us",
    "ma'am": "madam",
    "mayn't": "may not",
    "might've": "might have",
    "mightn't": "might not",
    "mightn't've": "might not have",
    "must've": "must have",
    "mustn't": "must not",
    "mustn't've": "must not have",
    "needn't": "need not",
    "needn't've": "need not have",
    "o'clock": "of the clock",
    "oughtn't": "ought not",
    "oughtn't've": "ought not have",
    "shan't": "shall not",
    "sha'n't": "shall not",
    "shan't've": "shall not have",
    "she'd": "she would",
    "she'd've": "she would have",
    "she'll": "she will",
    "she'll've": "she will have",
    "she's": "she is",
    "should've": "should have",
    "shouldn't": "should not",
    "shouldn't've": "should not have",
    "so've": "so have",
    "so's": "so as",
    "that'd": "that would",
    "that'd've": "that would have",
    "that's": "that is",
    "there'd": "there would",
    "there'd've": "there would have",
    "there's": "there is",
    "they'd": "they would",
    "they'd've": "they would have",
    "they'll": "they will",
    "they'll've": "they will have",
    "they're": "they are",
    "they've": "they have",
    "to've": "to have",
    "wasn't": "was not",
    "we'd": "we would",
    "we'd've": "we would have",
    "we'll": "we will",
    "we'll've": "we will have",
    "we're": "we are",
    "we've": "we have",
    "weren't": "were not",
    "what'll": "what will",
    "what'll've": "what will have",
    "what're": "what are",
    "what's": "what is",
    "what've": "what have",
    "when's": "when is",
    "when've": "when have",
    "where'd": "where did",
    "where's": "where is",
    "where've": "where have",
    "who'll": "who will",
    "who'll've": "who will have",
    "who's": "who is",
    "who've": "who have",
    "why's": "why is",
    "why've": "why have",
    "will've": "will have",
    "won't": "will not",
    "won't've": "will not have",
    "would've": "would have",
    "wouldn't": "would not",
    "wouldn't've": "would not have",
    "y'all": "you all",
    "y'all'd": "you all would",
    "y'all'd've": "you all would have",
    "y'all're": "you all are",
    "y'all've": "you all have",
    "you'd": "you would",
    "you'd've": "you would have",
    "you'll": "you will",
    "you'll've": "you will have",
    "you're": "you are",
    "you've": "you have"
    }


    q_decontracted = []

    for word in q.split():
        if word in contractions:
            word = contractions[word]

        q_decontracted.append(word)

    q = ' '.join(q_decontracted)
    q = q.replace("'ve", " have")
    q = q.replace("n't", " not")
    q = q.replace("'re", " are")
    q = q.replace("'ll", " will")
    
    # Removing HTML tags
    q = BeautifulSoup(q)
    q = q.get_text()
    
    # Remove punctuations
    pattern = re.compile('\W')
    q = re.sub(pattern, ' ', q).strip()

   # remove all kinds of whitespace
    q = re.sub(r'\s+', ' ', q).strip()
    
    return q

In [12]:
# apply the preprocess function on dataset 

new_df['question1'] = new_df['question1'].apply(preprocess)
new_df['question2'] = new_df['question2'].apply(preprocess)

In [13]:
# Create a new columns 'q1_len', q2_len char lenth of q1 and q2

new_df['q1_len'] = new_df['question1'].str.len() 
new_df['q2_len'] = new_df['question2'].str.len()

In [19]:
# create a new column of 'q1_num_words'  words in q1

new_df['q1_num_words'] = new_df['question1'].apply(lambda row: len(row.split(" ")))
new_df['q2_num_words'] = new_df['question2'].apply(lambda row: len(row.split(" ")))


In [20]:
def common_words(row):
    w1 = set(map(lambda word: word.lower().strip(), row['question1'].split(" ")))
    w2 = set(map(lambda word: word.lower().strip(), row['question2'].split(" ")))    
    return len(w1 & w2)

In [21]:
new_df['word_common'] = new_df.apply(common_words, axis=1)


In [22]:
def total_words(row):
    w1 = set(map(lambda word: word.lower().strip(), row['question1'].split(" ")))
    w2 = set(map(lambda word: word.lower().strip(), row['question2'].split(" ")))    
    return (len(w1) + len(w2))

In [23]:
new_df['word_total'] = new_df.apply(total_words, axis=1)


In [24]:
new_df['word_share'] = round(new_df['word_common']/new_df['word_total'],2)


# Advanced Features

In [25]:
import nltk

# nltk.download('stopwords')

In [26]:
import nltk
from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))
print(stop_words)

{'s', "i've", 'mightn', 'these', 'by', 'll', "they'll", "aren't", 'hasn', 'here', 'any', 'only', 'out', "couldn't", 'me', 'again', 'shouldn', 'do', 'o', 'or', "mustn't", 'should', "wouldn't", 'you', "that'll", "i'm", 'so', 'shan', 'than', "you'll", "shouldn't", 'isn', 'further', 'myself', 'while', 'very', "won't", 'himself', "mightn't", 'his', 'then', "we'd", 'don', 'in', "should've", 'their', 'themselves', 'who', 'below', "weren't", 'above', 'him', 'she', 'weren', 'they', 'd', "don't", 'we', 'now', 'the', 'a', 'aren', 'been', 'can', 'our', "haven't", 'with', 'hers', "i'll", 'whom', 'was', "shan't", 'just', "you've", 'for', "he'll", 'nor', "we'll", 'ain', 'which', 'until', 'ourselves', "it'll", 'into', "needn't", "she'll", "it's", 'before', 'that', 'has', 'have', "they're", 'between', 'wasn', 'all', 'couldn', 'your', 'up', "didn't", 'needn', 'after', "hadn't", 'such', "we're", 'during', 'is', 'no', 'does', 'had', 'hadn', 'down', 'how', 'about', 'my', 'of', 'on', 'some', "we've", 'he', 

In [27]:
# Advanced Features
from nltk.corpus import stopwords

def fetch_token_features(row):
    
    q1 = row['question1']
    q2 = row['question2']
    
    SAFE_DIV = 0.0001 

    STOP_WORDS = stopwords.words("english")
    
    token_features = [0.0]*8
    
    # Converting the Sentence into Tokens: 
    q1_tokens = q1.split()
    q2_tokens = q2.split()
    
    if len(q1_tokens) == 0 or len(q2_tokens) == 0:
        return token_features

    # Get the non-stopwords in Questions
    q1_words = set([word for word in q1_tokens if word not in STOP_WORDS])
    q2_words = set([word for word in q2_tokens if word not in STOP_WORDS])

    # get the stopwords in questions:
    q1_stops = set([word for word in q1_tokens if word in STOP_WORDS])
    q2_stops = set([word for word in q2_tokens if word in STOP_WORDS])

    # Get the common non-stopwords from question pair
    common_word_count = len(q1_words.intersection(q2_words))

    # Get the common stopwords from question pair
    common_stop_count = len(q1_stops.intersection(q2_stops))

    # Get the common tokens from question pair
    common_token_count = len(set(q1_tokens).intersection(set(q2_tokens)))


    token_features[0] = common_word_count / (min(len(q1_words), len(q2_words)) + SAFE_DIV)
    token_features[1] = common_word_count / (max(len(q1_words), len(q2_words)) + SAFE_DIV)
    token_features[2] = common_stop_count / (min(len(q1_stops), len(q2_stops)) + SAFE_DIV)
    token_features[3] = common_stop_count / (max(len(q1_stops), len(q2_stops)) + SAFE_DIV)
    token_features[4] = common_token_count / (min(len(q1_tokens), len(q2_tokens)) + SAFE_DIV)
    token_features[5] = common_token_count / (max(len(q1_tokens), len(q2_tokens)) + SAFE_DIV)

    # Last word of both question is same or not
    token_features[6] = int(q1_tokens[-1] == q2_tokens[-1])

    # First word of both question is same or not
    token_features[7] = int(q1_tokens[0] == q2_tokens[0])

    return token_features
    

In [28]:
token_features = new_df.apply(fetch_token_features, axis = 1)

In [29]:
new_df['cwc_min'] = list(map(lambda x: x[0], token_features))
new_df['cwc_max'] = list(map(lambda x: x[1], token_features))
new_df['csc_min'] = list(map(lambda x: x[2], token_features))
new_df['csc_max'] = list(map(lambda x: x[3], token_features))
new_df['ctc_min'] = list(map(lambda x: x[4], token_features))
new_df['ctc_max'] = list(map(lambda x: x[5], token_features))
new_df['last_word_eq'] = list(map(lambda x: x[6], token_features))
new_df['first_word_eq'] = list(map(lambda x: x[7], token_features))

# Length Based Features

In [30]:
# Length Based Features
import distance

def fetch_length_features(row):
    q1 = row['question1']
    q2 = row['question2']

    length_features = [0.0]*3

    # Converting the sentance into tokens:
    q1_tokens = q1.split()
    q2_tokens = q2.split()

    if len(q1_tokens) == 0 or len(q2_tokens) == 0:
        return length_features

    # Absolute length features
    length_features[0] = abs(len(q1_tokens) - len(q2_tokens))

    # Average Token Length of both Questions   
    length_features[1] = (len(q1_tokens) + len(q2_tokens))/2

    strs = list(distance.lcsubstrings(q1, q2))
    length_features[2] = len(strs[0]) / (min(len(q1), len(q2)) + 1)

    return length_features


In [31]:
length_features = new_df.apply(fetch_length_features, axis=1)

new_df['abs_len_diff'] = list(map(lambda x: x[0], length_features))
new_df['mean_len'] = list(map(lambda x: x[1], length_features))
new_df['longest_substr_ratio'] = list(map(lambda x: x[2], length_features))

# Fuzzy Features

In [32]:
# Fuzzy Features

from fuzzywuzzy import fuzz

def fetch_fuzzy_features(row):
    q1 = row['question1']
    q2 = row['question2']

    fuzzy_features = [0.0]*4

    # fuzz_ratio
    fuzzy_features[0] = fuzz.QRatio(q1, q2)  # Usage: This provides a similarity score by performing an exact character-by-character comparison of the two strings (case-sensitive).

    # fuzz_partial_ratio
    fuzzy_features[1] = fuzz.partial_ratio(q1, q2)   # Use: Yeh substring matching ke liye hota hai. Agar ek string choti hai aur wo badi string ke andar kahi bhi milti hai, toh yeh high score deta hai.


 # Use: Yeh words ko alphabetical order mein sort karke compare karta hai. Isse farq nahi padta ki words aage-piche likhe hain ya nahi (jaise "machine learning" aur "learning machine").
    # token_sort_ratio
    fuzzy_features[2] = fuzz.token_sort_ratio(q1, q2)   


# Usage: It ignores duplicate words and matches overlapping words without regard to order.
    # token_set_ratio
    fuzzy_features[3] = fuzz.token_set_ratio(q1, q2)  


    return fuzzy_features


In [33]:
fuzzy_features = new_df.apply(fetch_fuzzy_features, axis=1)

# Creating new feature columns for fuzzy features
new_df['fuzz_ratio'] = list(map(lambda x: x[0], fuzzy_features))
new_df['fuzz_partial_ratio'] = list(map(lambda x: x[1], fuzzy_features))
new_df['token_sort_ratio'] = list(map(lambda x: x[2], fuzzy_features))
new_df['token_set_ratio'] = list(map(lambda x: x[3], fuzzy_features))

# Perform EDA on new columns 

In [34]:
ques_df = new_df[['question1','question2']]


In [35]:
final_df = new_df.drop(columns=['id','qid1','qid2','question1','question2'])


## Word2vec

In [36]:
import numpy as np
from gensim.models import Word2Vec

# Select questions
ques_df = new_df[['question1', 'question2']].copy()


# 1. Tokenization

questions = list(ques_df['question1']) + list(ques_df['question2'])

tokenized_questions = [question.lower().split() for question in questions]

# 2. Train Word2Vec

w2v_model = Word2Vec(
    sentences=tokenized_questions,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    sg=1
)

# 3. Convert each question into a vector

def question_to_vector(question, model):

    words = question.lower().split()

    vectors = [model.wv[word] for word in words if word in model.wv]

    if len(vectors) == 0:
        return np.zeros(model.vector_size)

    return np.mean(vectors, axis=0)



# 4. Create vectors for all questions

question_vectors = np.array([question_to_vector(question, w2v_model) for question in questions])


# 5. Split Q1 and Q2

q1_arr, q2_arr = np.vsplit(question_vectors, 2)

print("Q1 shape:", q1_arr.shape)
print("Q2 shape:", q2_arr.shape)


Q1 shape: (30000, 100)
Q2 shape: (30000, 100)


In [37]:
temp_df1 = pd.DataFrame(q1_arr, index= ques_df.index)
temp_df2 = pd.DataFrame(q2_arr, index= ques_df.index)
temp_df = pd.concat([temp_df1, temp_df2], axis=1)

In [38]:
final_df = pd.concat([final_df, temp_df], axis=1)


In [78]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(final_df.iloc[:,1:].values,final_df.iloc[:,0].values,test_size=0.2,random_state=1)

In [40]:
final_df.shape

(30000, 223)

In [79]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# ensemble_stacking:

In [30]:
# ensemble_stacking 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import SGDClassifier, LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import NuSVC
from sklearn.ensemble import StackingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


# 2. Base Models (Individual Classifiers)
clf_lr  = SGDClassifier(penalty='l2', eta0=0.01, loss='log_loss', max_iter=200,
                        random_state=42, learning_rate='adaptive', alpha=0.0001)
clf_dtr = DecisionTreeClassifier(random_state=4, splitter="best",
                                 min_impurity_decrease=0.0001, min_samples_leaf=20,
                                 max_features=220, ccp_alpha=0.001)
clf_knn = KNeighborsClassifier(n_neighbors=5, weights='uniform', n_jobs=-1, metric='euclidean')
clf_svm = NuSVC(max_iter=100, kernel='rbf', nu=0.2, probability=True)
clf_nb  = GaussianNB(var_smoothing=1e-9)


# 3. Stacking Ensemble
ensemble_stacking = StackingClassifier(
    estimators=[
        ('knn', clf_knn),
        ('nb',  clf_nb),
        ('svm', clf_svm),
        ('logistic', clf_lr),
        ('dtr', clf_dtr),
    ],
    final_estimator=LogisticRegression(max_iter=100, c = 0.1),
    stack_method='auto',   # uses predict_proba if available, else decision_function, else predict
    cv=5,                  
    passthrough=False,     # if True, original features are also fed to the meta-learner
    n_jobs=-1
)


# Train & Evaluate
ensemble_stacking.fit(X_train_sc, y_train)

print("--- Model Performance Comparison ---")
# Detailed Report for Stacking Model
print("\n--- Detailed Report for Stacking Model ---")
y_pred_stack = ensemble_stacking.predict(X_test_sc)
print("accuracy_score", accuracy_score(y_test, y_pred_stack))
print("classification_report", classification_report(y_test, y_pred_stack))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_stack))


In [ ]:
# train accuracy
y_train_pres = ensemble_stacking.predict(X_train_sc)
train_acc = accuracy_score(y_train, y_train_pres)

# Testing accuracy
y_test_pred = ensemble_stacking.predict(X_test_sc)
test_acc = accuracy_score(y_test, y_test_pred)

print("Training Accuracy:", train_acc)
print("Testing Accuracy:", test_acc)
print("Accuracy Gap:", train_acc - test_acc)

# GridSearch Stracking Ensemble 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from sklearn.linear_model import SGDClassifier, LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import NuSVC
from sklearn.ensemble import StackingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# --- 1. Base Models (Individual Classifiers) ---
clf_lr  = SGDClassifier(penalty='l2', eta0=0.01, loss='log_loss', max_iter=200,
                        random_state=42, learning_rate='adaptive', alpha=0.0001)
clf_dtr = DecisionTreeClassifier(random_state=4, splitter="best",
                                 min_impurity_decrease=0.0001, min_samples_leaf=20,
                                 max_features=2, ccp_alpha=0.001) # Note: max_features adjusted for 2D plot demo if needed
clf_knn = KNeighborsClassifier(n_neighbors=5, weights='uniform', n_jobs=-1, metric='euclidean')
clf_svm = NuSVC(max_iter=100, kernel='rbf', nu=0.2, probability=True)
clf_nb  = GaussianNB(var_smoothing=1e-9)

# -2. Stacking Ensemble
ensemble_stacking = StackingClassifier(
    estimators=[
        ('knn', clf_knn),
        ('nb',  clf_nb),
        ('svm', clf_svm),
        ('logistic', clf_lr),
        ('dtr', clf_dtr),
    ],
    final_estimator=LogisticRegression(max_iter=100, C=0.1, solver='lbfgs'),
    stack_method='auto',   
    cv=5,                     
    passthrough=False,       
    n_jobs=-1
)

# Assume X_train_sc and y_train are already defined and are 2D (2 features for easy plotting)
ensemble_stacking.fit(X_train_sc, y_train)

# --- 3. Plotting the Decision Boundary ---
plt.figure(figsize=(10, 6))

# Create mesh grid over feature space
x_min, x_max = X_train_sc[:, 0].min() - 1, X_train_sc[:, 0].max() + 1
y_min, y_max = X_train_sc[:, 1].min() - 1, X_train_sc[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.1),
                     np.arange(y_min, y_max, 0.1))

# Predict over the grid
Z = ensemble_stacking.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

# Plot contour map
cmap_light = ListedColormap(['#FFAAAA', '#AAFFAA', '#AAAAFF'])
cmap_bold = ListedColormap(['#FF0000', '#00FF00', '#0000FF'])

plt.contourf(xx, yy, Z, alpha=0.3, cmap=cmap_light)

# Plot training points
plt.scatter(X_train_sc[:, 0], X_train_sc[:, 1], c=y_train, cmap=cmap_bold, edgecolor='k', s=40)
plt.title("Stacking Classifier Decision Boundary (with Logistic Regression Meta-Model)")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.show()

# --- 4. Evaluation ---
print("--- Model Performance Comparison ---")
y_pred_stack = ensemble_stacking.predict(X_test_sc)
print("accuracy_score", accuracy_score(y_test, y_pred_stack))
print("classification_report:\n", classification_report(y_test, y_pred_stack))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_stack))

In [69]:
import xgboost as xgb
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


# 3. Initialize the XGBoost Classifier with key hyperparameters
xgb = xgb.XGBClassifier(
    n_estimators=200,  # Number of trees (boosting rounds)
    learning_rate=0.05,  # Step size shrinkage (eta) to prevent overfitting
    max_depth=7,  # Maximum depth of each tree
    random_state=3,
    eval_metric='logloss',  # Evaluation metric for training progress
    n_jobs = -1
)

# 4. Train (fit) the model on the training data
print('Training the XGBoost model...')
xgb.fit(X_train_sc, y_train)

# 5. Make predictions on the test set
y_pred = xgb.predict(X_test_sc)

# 6. Evaluate the model performance
accuracy = accuracy_score(y_test, y_pred)
print(f'\nModel Accuracy: {accuracy * 100:.2f}%\n')

print('Classification Report:', classification_report(y_test, y_pred))

print("matrix: \n", confusion_matrix(y_test, y_pred))

Training the XGBoost model...

Model Accuracy: 79.23%

Classification Report:               precision    recall  f1-score   support

           0       0.83      0.85      0.84      3812
           1       0.72      0.70      0.71      2188

    accuracy                           0.79      6000
   macro avg       0.78      0.77      0.77      6000
weighted avg       0.79      0.79      0.79      6000

matrix: 
 [[3225  587]
 [ 659 1529]]


In [70]:

# train accuracy
y_train_pres = xgb.predict(X_train_sc)
train_acc = accuracy_score(y_train, y_train_pres)

# Testing accuracy
y_test_pred = xgb.predict(X_test_sc)
test_acc = accuracy_score(y_test, y_test_pred)

print("Training Accuracy:", train_acc)
print("Testing Accuracy:", test_acc)
print("Accuracy Gap:", train_acc - test_acc)



Training Accuracy: 0.9370416666666667
Testing Accuracy: 0.7923333333333333
Accuracy Gap: 0.14470833333333333


In [ ]:
Training the XGBoost model...

Model Accuracy: 79.23%

Classification Report:               precision    recall  f1-score   support

           0       0.83      0.85      0.84      3812
           1       0.72      0.70      0.71      2188

    accuracy                           0.79      6000
   macro avg       0.78      0.77      0.77      6000
weighted avg       0.79      0.79      0.79      6000

matrix: 
 [[3225  587]
 [ 659 1529]]
----------------------------------------------------------------------

Training Accuracy: 0.9370416666666667
Testing Accuracy: 0.7923333333333333
Accuracy Gap: 0.14470833333333333

In [68]:
import xgboost as xgb
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 1. Initialize the base XGBoost Classifier
xgb_clf = xgb.XGBClassifier(
    random_state=3,
    eval_metric='logloss',
    n_jobs=-1
)

# 2. Define the parameter grid you want to search over
param_grid = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.05, 0.1, 0.2],
    'max_depth': [3, 5, 7]
}

# 3. Set up GridSearchCV
print('Setting up GridSearchCV...')
grid_search = GridSearchCV(
    estimator=xgb_clf,
    param_grid=param_grid,
    scoring='accuracy',
    cv=3,          # 3-fold cross-validation (you can increase to 5 if dataset allows)
    n_jobs=-1,     # Uses all available CPU cores
    verbose=1      # Prints progress updates to the console
)

# 4. Train (fit) the grid search on the training data
print('Training the XGBoost model with GridSearchCV...')
grid_search.fit(X_train_sc, y_train)

# 5. Output the best hyperparameters found
print('\nBest Parameters Found:')
print(grid_search.best_params_)

# 6. Extract the best estimator model
best_xgb = grid_search.best_estimator_

# 7. Make predictions on the test set using the best model
y_pred = best_xgb.predict(X_test_sc)

# 8. Evaluate the model performance
accuracy = accuracy_score(y_test, y_pred)
print(f'\nModel Accuracy (Best Model): {accuracy * 100:.2f}%\n')

print('Classification Report:\n', classification_report(y_test, y_pred))
print('Confusion Matrix:\n', confusion_matrix(y_test, y_pred))

Setting up GridSearchCV...
Training the XGBoost model with GridSearchCV...
Fitting 3 folds for each of 27 candidates, totalling 81 fits

Best Parameters Found:
{'learning_rate': 0.05, 'max_depth': 7, 'n_estimators': 200}

Model Accuracy (Best Model): 79.23%

Classification Report:
               precision    recall  f1-score   support

           0       0.83      0.85      0.84      3812
           1       0.72      0.70      0.71      2188

    accuracy                           0.79      6000
   macro avg       0.78      0.77      0.77      6000
weighted avg       0.79      0.79      0.79      6000

Confusion Matrix:
 [[3225  587]
 [ 659 1529]]


In [46]:
import re
from bs4 import BeautifulSoup
import distance
from fuzzywuzzy import fuzz
import pickle
import numpy as np
from gensim.models import Word2Vec

# Word2Vec model load karna (Ensure karein ki file saved ho)
w2v_model = Word2Vec.load("../w2v_model.bin")


def test_common_words(q1, q2):
    w1 = set(map(lambda word: word.lower().strip(), q1.split(" ")))
    w2 = set(map(lambda word: word.lower().strip(), q2.split(" ")))
    return len(w1 & w2)


def test_total_words(q1, q2):
    w1 = set(map(lambda word: word.lower().strip(), q1.split(" ")))
    w2 = set(map(lambda word: word.lower().strip(), q2.split(" ")))
    return (len(w1) + len(w2))


def test_fetch_token_features(q1, q2):
    SAFE_DIV = 0.0001
    
    # Stopwords ko pickle se load karna
    STOP_WORDS = pickle.load(open('../stopwords.pkl','rb'))

    token_features = [0.0] * 8

    q1_tokens = q1.split()
    q2_tokens = q2.split()

    if len(q1_tokens) == 0 or len(q2_tokens) == 0:
        return token_features

    q1_words = set([word for word in q1_tokens if word not in STOP_WORDS])
    q2_words = set([word for word in q2_tokens if word not in STOP_WORDS])

    q1_stops = set([word for word in q1_tokens if word in STOP_WORDS])
    q2_stops = set([word for word in q2_tokens if word in STOP_WORDS])

    common_word_count = len(q1_words.intersection(q2_words))
    common_stop_count = len(q1_stops.intersection(q2_stops))
    common_token_count = len(set(q1_tokens).intersection(set(q2_tokens)))

    token_features[0] = common_word_count / (min(len(q1_words), len(q2_words)) + SAFE_DIV)
    token_features[1] = common_word_count / (max(len(q1_words), len(q2_words)) + SAFE_DIV)
    token_features[2] = common_stop_count / (min(len(q1_stops), len(q2_stops)) + SAFE_DIV)
    token_features[3] = common_stop_count / (max(len(q1_stops), len(q2_stops)) + SAFE_DIV)
    token_features[4] = common_token_count / (min(len(q1_tokens), len(q2_tokens)) + SAFE_DIV)
    token_features[5] = common_token_count / (max(len(q1_tokens), len(q2_tokens)) + SAFE_DIV)

    token_features[6] = int(q1_tokens[-1] == q2_tokens[-1]) if q1_tokens and q2_tokens else 0
    token_features[7] = int(q1_tokens[0] == q2_tokens[0]) if q1_tokens and q2_tokens else 0

    return token_features


def test_fetch_length_features(q1, q2):
    length_features = [0.0] * 3

    q1_tokens = q1.split()
    q2_tokens = q2.split()

    if len(q1_tokens) == 0 or len(q2_tokens) == 0:
        return length_features

    length_features[0] = abs(len(q1_tokens) - len(q2_tokens))
    length_features[1] = (len(q1_tokens) + len(q2_tokens)) / 2

    strs = list(distance.lcsubstrings(q1, q2))
    length_features[2] = len(strs[0]) / (min(len(q1), len(q2)) + 1) if strs else 0.0

    return length_features


def test_fetch_fuzzy_features(q1, q2):
    fuzzy_features = [0.0] * 4

    fuzzy_features[0] = fuzz.QRatio(q1, q2)
    fuzzy_features[1] = fuzz.partial_ratio(q1, q2)
    fuzzy_features[2] = fuzz.token_sort_ratio(q1, q2)
    fuzzy_features[3] = fuzz.token_set_ratio(q1, q2)

    return fuzzy_features


def preprocess(q):
    q = str(q).lower().strip()

    q = q.replace('%', ' percent')
    q = q.replace('$', ' dollar ')
    q = q.replace('₹', ' rupee ')
    q = q.replace('€', ' euro ')
    q = q.replace('@', ' at ')
    q = q.replace('[math]', '')

    q = q.replace(',000,000,000 ', 'b ')
    q = q.replace(',000,000 ', 'm ')
    q = q.replace(',000 ', 'k ')
    q = re.sub(r'([0-9]+)000000000', r'\1b', q)
    q = re.sub(r'([0-9]+)000000', r'\1m', q)
    q = re.sub(r'([0-9]+)000', r'\1k', q)

    contractions = {
        "ain't": "am not", "aren't": "are not", "can't": "can not", "can't've": "can not have",
        "'cause": "because", "could've": "could have", "couldn't": "could not", "didn't": "did not",
        "doesn't": "does not", "don't": "do not", "hadn't": "had not", "hasn't": "has not",
        "haven't": "have not", "he'd": "he would", "he'll": "he will", "he's": "he is",
        "i'd": "i would", "i'll": "i will", "i'm": "i am", "i've": "i have", "isn't": "is not",
        "it's": "it is", "let's": "let us", "she'd": "she would", "she'll": "she will",
        "she's": "she is", "should've": "should have", "shouldn't": "should not", "that's": "that is",
        "there's": "there is", "they'd": "they would", "they'll": "they will", "they're": "they are",
        "they've": "they have", "wasn't": "was not", "we'd": "we would", "we're": "we are",
        "we've": "we have", "weren't": "were not", "what'll": "what will", "what're": "what are",
        "what's": "what is", "when's": "when is", "where's": "where is", "who's": "who is",
        "why's": "why is", "won't": "will not", "would've": "would have", "wouldn't": "would not",
        "you'd": "you would", "you'll": "you will", "you're": "you are", "you've": "you have"
    }

    q_decontracted = []
    for word in q.split():
        if word in contractions:
            word = contractions[word]
        q_decontracted.append(word)

    q = ' '.join(q_decontracted)
    q = q.replace("'ve", " have")
    q = q.replace("n't", " not")
    q = q.replace("'re", " are")
    q = q.replace("'ll", " will")

    q = BeautifulSoup(q, "html.parser").get_text()
    pattern = re.compile(r'\W')
    q = re.sub(pattern, ' ', q).strip()
    q = re.sub(r'\s+', ' ', q).strip()

    return q


def question_to_vector(question, model):
    words = question.lower().split()
    vectors = [model.wv[word] for word in words if word in model.wv]
    if len(vectors) == 0:
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)


def query_point_creator(q1, q2):
    input_query = []

    # preprocess
    q1 = preprocess(q1)
    q2 = preprocess(q2)

    # fetch basic features (7 features)
    input_query.append(len(q1))
    input_query.append(len(q2))
    input_query.append(len(q1.split(" ")))
    input_query.append(len(q2.split(" ")))
    
    common_w = test_common_words(q1, q2)
    total_w = test_total_words(q1, q2)
    input_query.append(common_w)
    input_query.append(total_w)
    input_query.append(round(common_w / total_w, 2) if total_w > 0 else 0.0)

    # fetch token features (8 features)
    token_features = test_fetch_token_features(q1, q2)
    input_query.extend(token_features)

    # fetch length based features (3 features)
    length_features = test_fetch_length_features(q1, q2)
    input_query.extend(length_features)

    # fetch fuzzy features (4 features)
    fuzzy_features = test_fetch_fuzzy_features(q1, q2)
    input_query.extend(fuzzy_features)
    # Total len of input_query is now 22 features

    # Word2Vec vectors for q1 and q2 (Har ek question ka 100-dim vector banega)
    q1_vec = question_to_vector(q1, w2v_model)
    q2_vec = question_to_vector(q2, w2v_model)

    # Sabhi features ko horizontal stack karna (22 + 100 + 100 = 222 features total)
    return np.hstack((np.array(input_query).reshape(1, 22), q1_vec.reshape(1, -1), q2_vec.reshape(1, -1)))

In [75]:
# 1. Test karne ke liye do sample questions dein

test_q1 = 'What is the difference between AI and ML?'
test_q2 = 'What are the advantages of using Python?'


try:
    # 2. Query point creator function ko call karein
    test_query = query_point_creator(test_q1, test_q2)
    print("Query point created successfully!")
    print("Shape of test query features:", test_query.shape) # Yeh (1, 222) aana chahiye
    
    # 3. Scaler transform test karein
    test_scaled = scaler.transform(test_query)
    print("Scaler successfully apply ho gaya!")
    
    # 4. Model prediction test karein
    prediction = xgb.predict(test_scaled)
    print("Model prediction successfully run!")
    print("Prediction Result (1 = Duplicate, 0 = Different):", prediction[0])

except Exception as e:
    print("There is Error:", e)

Query point created successfully!
Shape of test query features: (1, 222)
Scaler successfully apply ho gaya!
Model prediction successfully run!
Prediction Result (1 = Duplicate, 0 = Different): 0


py -3.12 -m venv venv 